# Attribute Recognition & Person Re-Identification — v2**Giacomo Lazzerini · Dario Fabiani — University of Trento**Second iteration of the *Quadruplet network* assignment. Same two tasks:1. **Attribute recognition** — predict 29 attributes for each image in `test/`.2. **Person re-identification** — for each image in `queries/`, rank the images of   `test/` by identity similarity.### What changed with respect to v1| | v1 | v2 ||---|---|---|| Batch construction | 1 random quadruplet per sample | **PK sampler** (P identities × K images) || Metric loss | quadruplet on random negatives | **quadruplet with in-batch hard mining** || Forward passes / step | 4 | **1** || Embedding head | raw pooled features | **BNNeck** (triplet before BN, classifier after) || Identity supervision | none | **ID classification head** + label smoothing || Attribute loss | unweighted BCE / CE | **class-weighted**, imbalance aware || Input resolution | 224×224 (aspect ratio destroyed) | **256×128** || Pooling | average | **GeM** (generalised mean) || Optimiser | SGD, wd silently disabled | **AdamW + warmup + cosine** || Precision | fp32 | **AMP (bf16/fp16) + channels_last** || Test-time | single pass | **flip augmentation averaging** || Metrics | mean per-attribute accuracy | **+ mAP, CMC@1/5/10, per-attribute F1 / balanced acc** || Checkpointing | saved a *randomly initialised* model | **best-on-val-mAP, resumable** |Everything is sized to fit an **8 GB laptop GPU** (RTX 4070 Laptop and similar);see the *Hardware budget* section for the numbers.

## 0. Setup

In [ ]:
import os
import sys
import json
import math
import random
import zipfile
from collections import defaultdict
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import train_test_split

IN_COLAB = 'google.colab' in sys.modules
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch', torch.__version__, '| device', DEVICE)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name} | {p.total_memory / 2**30:.1f} GiB | SM {p.major}.{p.minor}')

In [ ]:
DATA_ROOT = os.environ.get('DATA_ROOT', '/content/data' if IN_COLAB else './data')
DATASET_ZIP = os.environ.get('DATASET_ZIP', 'dataset.zip')

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    if IN_COLAB and not os.path.exists(DATASET_ZIP):
        from google.colab import drive
        drive.mount('/content/gdrive')
        DATASET_ZIP = '/content/gdrive/MyDrive/DEEP_LEARNING_PROJECT/dataset.zip'
    print(f'extracting {DATASET_ZIP} -> {DATA_ROOT}')
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(DATA_ROOT)

for sub in ('train', 'test', 'queries'):
    print(f'{sub:8s} {len(os.listdir(os.path.join(DATA_ROOT, sub)))} images')

### ConfigurationOne dataclass drives the whole notebook. The defaults are the **8 GB preset**:ResNet50 with `last_stride=1`, 256×128 inputs, AMP, and a PK batch of`P*K = 16*4 = 64` images — which is a *single* forward pass, not four.Presets you can switch to:* `Config(backbone='resnet18', p_identities=24)` — fast, ~2 GB, good for debugging.* `Config(backbone='resnet50', last_stride=1, p_identities=16, k_instances=4)` — default.* `Config(..., grad_accum=2)` — emulates a 128-image batch without the memory.

In [ ]:
@dataclass
class Config:
    # data
    data_root: str = DATA_ROOT
    image_size: tuple = (256, 128)          # (H, W); source images are 64x128
    val_id_fraction: float = 0.28           # identities held out for validation
    seed: int = 42

    # batch: P identities x K images each -> P*K images per step.
    # P=12, K=4 (not P=4, K=12): what matters for mining is how many *negative
    # identities* each anchor sees, and K only needs to be large enough to give a
    # meaningful hardest positive.
    p_identities: int = 12
    k_instances: int = 4
    num_workers: int = min(8, os.cpu_count() or 2)
    eval_batch_size: int = 128

    # model
    backbone: str = 'resnet50'               # resnet18 / resnet34 / resnet50
    ibn: bool = False                        # IBN-a variant (needs timm)
    last_stride: int = 1                     # 1 -> 2x finer feature map (re-ID trick)
    embedding_dim: int = 512                 # 0 = use the backbone dim directly
    pooling: str = 'gem'                     # gem | avg
    dropout: float = 0.2
    bnneck: bool = True

    # losses
    metric_loss: str = 'quadruplet'          # quadruplet | circle
    lambda_attr: float = 1.0
    lambda_quadruplet: float = 1.0
    lambda_id: float = 1.0
    margin1: float = 0.3                     # anchor-positive vs anchor-negative
    margin2: float = 0.15                    # anchor-positive vs negative-negative
    soft_margin: bool = True                 # softplus instead of hinge
    label_smoothing: float = 0.1
    attr_class_weights: bool = True
    attr_weight_cap: float = 20.0
    circle_m: float = 0.25                   # only used when metric_loss='circle'
    circle_gamma: float = 64.0

    # optimisation
    epochs: int = 30
    lr: float = 3e-4
    lr_backbone_mult: float = 0.1            # pretrained trunk moves slower
    weight_decay: float = 5e-4
    warmup_epochs: int = 3
    grad_accum: int = 1
    grad_clip: float = 10.0
    amp: bool = True
    channels_last: bool = True

    # early stopping
    patience: int = 8                        # evaluations without improvement; 0 = off
    min_epochs: int = 10                     # never stop before this

    # evaluation
    map_rank: int = 20                       # the assignment ranks the top-20
    flip_test: bool = True
    re_rank: bool = False                    # k-reciprocal re-ranking at test time
    re_rank_k1: int = 20
    re_rank_k2: int = 6
    re_rank_lambda: float = 0.3
    eval_every: int = 1
    quick_run: bool = False                  # tiny subset, for smoke tests

    # bookkeeping
    out_dir: str = 'runs/v2'
    checkpoint: str = 'runs/v2/best.pt'

    @property
    def batch_size(self):
        return self.p_identities * self.k_instances


cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(cfg.seed)
torch.backends.cudnn.benchmark = True
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=1))

## 1. Annotations`annotations_train.csv` gives 27 attributes for each of the 751 identities, with1-based labels. We shift them to 0-based and derive two extra attributes(`upmulti` / `downmulti`, set when no single up/down colour is active), for 29attributes in total. `age` is the only multi-class one (4 classes).Note that the attribute labels are **per identity**, not per image: all images ofa person share its attributes. This means the effective number of independentattribute samples is 751, not 12,989 — which is exactly why the attribute headsoverfit quickly and why the train/val split must be done **by identity**.

In [ ]:
csv = pd.read_csv(os.path.join(cfg.data_root, 'annotations_train.csv'))

ANNOTATED = csv.columns[1:].tolist()                     # 27
DERIVED = ['upmulti', 'downmulti']
ATTRIBUTES = ANNOTATED + DERIVED                         # 29
N_CLASSES = {a: int(csv[a].max()) for a in ANNOTATED}    # age -> 4, rest -> 2
# 1 output = binary (BCEWithLogits); n>2 outputs = multi-class (CrossEntropy)
ATTR_MAP = [1 if N_CLASSES.get(a, 2) == 2 else N_CLASSES[a] for a in ATTRIBUTES]

files = sorted(f for f in os.listdir(os.path.join(cfg.data_root, 'train')) if f.endswith('.jpg'))
df = pd.DataFrame({
    'image_name': files,
    'ID': [int(f.split('_')[0]) for f in files],
    'cam': [int(f.split('_')[1][1]) for f in files],
})
df = df.merge(csv, left_on='ID', right_on='id', how='left').drop(columns='id')
assert df[ANNOTATED].notna().all().all()
df[ANNOTATED] = df[ANNOTATED].astype(int) - 1

UP_COLOURS = [a for a in ANNOTATED if a.startswith('up')]
DOWN_COLOURS = [a for a in ANNOTATED if a.startswith('down') and a != 'down']
df['upmulti'] = (df[UP_COLOURS].sum(axis=1) == 0).astype(int)
df['downmulti'] = (df[DOWN_COLOURS].sum(axis=1) == 0).astype(int)
df = df[['image_name', 'ID', 'cam'] + ATTRIBUTES]

print(f'{len(df)} images, {df.ID.nunique()} identities, {len(ATTRIBUTES)} attributes')
print('attr_map:', ATTR_MAP)
df.head(3)

### How imbalanced are the attributes?This is the part v1 ignored. `hat` is positive on 2.7% of the images and `age` has82% of its samples in a single class, so a head that always answers with themajority class scores 97% accuracy while being useless. Two consequences, bothhandled below:* the loss gets **per-class weights**, so rare attributes actually produce gradient;* the report adds **balanced accuracy and F1**, so we can tell learning from  guessing the majority class.

In [ ]:
stats = []
for a in ATTRIBUTES:
    v = df[a]
    if ATTR_MAP[ATTRIBUTES.index(a)] == 1:
        pos = v.mean()
        stats.append((a, 'binary', f'{pos * 100:5.1f}% positive',
                      f'{max(pos, 1 - pos) * 100:5.1f}%'))
    else:
        counts = v.value_counts(normalize=True).sort_index()
        stats.append((a, f'{len(counts)}-class',
                      ' / '.join(f'{c * 100:.0f}%' for c in counts),
                      f'{counts.max() * 100:5.1f}%'))

imbalance = pd.DataFrame(stats, columns=['attribute', 'type', 'distribution',
                                         'majority-class accuracy'])
display(imbalance)

ax = df[[a for a in ATTRIBUTES if ATTR_MAP[ATTRIBUTES.index(a)] == 1]].mean().sort_values().plot.barh(
    figsize=(6, 8), title='positive rate per binary attribute')
ax.axvline(0.5, color='k', lw=.5)
plt.tight_layout(); plt.show()

### SplitsIdentities are split first, then the validation identities are split into a*query* and a *gallery* half so we can measure re-ID on held-out people. Querieswhose identity has no gallery image are dropped — their AP is 0 by constructionand would only depress the metric with noise.

In [ ]:
ids = sorted(df.ID.unique())
train_ids, val_ids = train_test_split(ids, test_size=cfg.val_id_fraction,
                                      random_state=cfg.seed)

train_df = df[df.ID.isin(train_ids)].reset_index(drop=True)
val_df = df[df.ID.isin(val_ids)].reset_index(drop=True)

if cfg.quick_run:                       # smoke test on a handful of identities
    train_df = train_df[train_df.ID.isin(train_ids[:40])].reset_index(drop=True)
    val_df = val_df[val_df.ID.isin(val_ids[:30])].reset_index(drop=True)

# the real submission ratio between queries and gallery
q_ratio = round(len(os.listdir(os.path.join(cfg.data_root, 'queries'))) /
                len(os.listdir(os.path.join(cfg.data_root, 'test'))), 2)

gallery_df, query_df = train_test_split(val_df, test_size=q_ratio,
                                        stratify=val_df.ID, random_state=cfg.seed)
gallery_df = gallery_df.reset_index(drop=True)
query_df = query_df[query_df.ID.isin(set(gallery_df.ID))].reset_index(drop=True)

# contiguous 0..N-1 identity labels for the ID classification head
ID2LABEL = {i: k for k, i in enumerate(sorted(train_df.ID.unique()))}
train_df['label'] = train_df.ID.map(ID2LABEL)
NUM_TRAIN_IDS = len(ID2LABEL)

print(f'train   {len(train_df):5d} images / {NUM_TRAIN_IDS} ids')
print(f'val     {len(val_df):5d} images / {val_df.ID.nunique()} ids')
print(f'  query {len(query_df):5d}  gallery {len(gallery_df):5d}  (ratio {q_ratio})')

## 2. Dataset and PK samplerThe dataset is deliberately dumb: one image, its labels, its identity. All thestructure that the quadruplet loss needs comes from the **sampler**.`PKSampler` yields batches made of `P` identities × `K` images each. Inside such abatch every sample has `K-1` positives and `(P-1)*K` negatives available, so theloss can *mine* the hard ones instead of hoping a random draw is informative.This is the standard triplet-with-batch-hard recipe (Hermans et al., 2017) and itis the single biggest accuracy difference with respect to v1 — which drew onerandom positive and two random negatives per anchor, needed four forward passes toembed them, and after a few epochs those random negatives were almost all alreadysatisfying the margin, contributing exactly zero gradient.

In [ ]:
class MarketDataset(Dataset):
    """labelled=True  -> (image, attributes, id_label)
       labelled=False -> image        (iterates sorted(os.listdir(root)))"""

    def __init__(self, root, dataframe=None, transform=None, labelled=True):
        self.root = root
        self.transform = transform
        self.labelled = labelled

        if labelled:
            self.frame = dataframe.reset_index(drop=True)
            self.names = self.frame.image_name.tolist()
            self.attrs = self.frame[ATTRIBUTES].to_numpy(np.int64)
            self.ids = self.frame.ID.to_numpy()
            self.labels = (self.frame.label.to_numpy()
                           if 'label' in self.frame else np.zeros(len(self.frame), np.int64))
        else:
            self.frame = None
            self.names = sorted(f for f in os.listdir(root) if f.endswith('.jpg'))

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.names[i])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if not self.labelled:
            return img
        return img, torch.from_numpy(self.attrs[i]), int(self.labels[i])


class PKSampler(Sampler):
    """Yields indices so that each batch holds P identities x K images."""

    def __init__(self, labels, p_identities, k_instances, seed=0):
        self.labels = np.asarray(labels)
        self.P, self.K = p_identities, k_instances
        self.by_label = defaultdict(list)
        for idx, l in enumerate(self.labels):
            self.by_label[l].append(idx)
        self.labels_unique = sorted(self.by_label)
        self.epoch = 0
        self.seed = seed
        # every identity contributes floor(n/K)*K samples per epoch
        self.length = sum(max(len(v) // self.K, 1) for v in self.by_label.values()) * self.K
        self.length = (self.length // self.batch) * self.batch

    @property
    def batch(self):
        return self.P * self.K

    def __len__(self):
        return self.length

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        # build, for each identity, a list of K-sized chunks of its images
        chunks = defaultdict(list)
        for l, idxs in self.by_label.items():
            idxs = idxs[:]
            rng.shuffle(idxs)
            if len(idxs) < self.K:                      # sample with replacement
                idxs = [rng.choice(idxs) for _ in range(self.K)]
            for s in range(0, len(idxs) - self.K + 1, self.K):
                chunks[l].append(idxs[s:s + self.K])

        available = [l for l in self.labels_unique if chunks[l]]
        order = []
        while len(available) >= self.P:
            picked = rng.sample(available, self.P)
            for l in picked:
                order.extend(chunks[l].pop())
                if not chunks[l]:
                    available.remove(l)
        return iter(order[:self.length])

### Augmentation`RandomErasing` (Zhong et al., 2017) simulates occlusion and is worth a couple ofmAP points on Market-1501; padding + crop gives small translations withoutchanging the resolution. Note what v1 did instead: `Resize(224, 224)` followed by`RandomCrop(32, padding=4)`, i.e. it trained on random 32×32 patches of anupscaled pedestrian.

In [ ]:
NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tfms = T.Compose([
    T.Resize(cfg.image_size, interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10, padding_mode='reflect'),
    T.RandomCrop(cfg.image_size),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.15, hue=0.02),
    T.ToTensor(),
    NORM,
    # hyper-parameters from Zhong et al. (arXiv:2002.11371), applied after Normalize
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0),
])

eval_tfms = T.Compose([
    T.Resize(cfg.image_size, interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    NORM,
])

fig, axs = plt.subplots(1, 6, figsize=(9, 3))
raw = Image.open(os.path.join(cfg.data_root, 'train', train_df.image_name[0])).convert('RGB')
inv = lambda t: (t * torch.tensor([0.229, 0.224, 0.225])[:, None, None]
                 + torch.tensor([0.485, 0.456, 0.406])[:, None, None]).clamp(0, 1).permute(1, 2, 0)
axs[0].imshow(raw); axs[0].set_title('raw', fontsize=8)
for ax in axs[1:]:
    ax.imshow(inv(train_tfms(raw)))
for ax in axs:
    ax.axis('off')
plt.suptitle('training augmentations', fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
train_ds = MarketDataset(os.path.join(cfg.data_root, 'train'), train_df, train_tfms)
val_ds = MarketDataset(os.path.join(cfg.data_root, 'train'), val_df, eval_tfms)
gallery_ds = MarketDataset(os.path.join(cfg.data_root, 'train'), gallery_df, eval_tfms)
query_ds = MarketDataset(os.path.join(cfg.data_root, 'train'), query_df, eval_tfms)

test_ds = MarketDataset(os.path.join(cfg.data_root, 'test'), transform=eval_tfms, labelled=False)
queries_ds = MarketDataset(os.path.join(cfg.data_root, 'queries'), transform=eval_tfms,
                           labelled=False)

loader_kw = dict(num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(),
                 persistent_workers=cfg.num_workers > 0)

sampler = PKSampler(train_df.label.to_numpy(), cfg.p_identities, cfg.k_instances, cfg.seed)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler,
                          drop_last=True, **loader_kw)

# eval loaders: never shuffled, never drop_last - the ground truth is positional
val_loader = DataLoader(val_ds, cfg.eval_batch_size, shuffle=False, **loader_kw)
gallery_loader = DataLoader(gallery_ds, cfg.eval_batch_size, shuffle=False, **loader_kw)
query_loader = DataLoader(query_ds, cfg.eval_batch_size, shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, cfg.eval_batch_size, shuffle=False, **loader_kw)
queries_loader = DataLoader(queries_ds, cfg.eval_batch_size, shuffle=False, **loader_kw)

print(f'{len(train_loader)} train steps/epoch of {cfg.batch_size} images '
      f'({cfg.p_identities} ids x {cfg.k_instances})')
imgs, attrs, labels = next(iter(train_loader))
print('batch', tuple(imgs.shape), '| unique ids in batch:', len(set(labels.tolist())))

## 3. Model```image ──▶ ResNet trunk (last_stride=1) ──▶ GeM pool ──▶ f_trunk                                                         │                            ┌────────────────────────────┤                            │                            │                     embedding (Linear)            (metric branch)                            │                            │                        BNNeck (BN1d)  ──▶ f_bn          f_emb ──▶ quadruplet loss                            │              ┌─────────────┴──────────────┐       ID classifier (751)          29 attribute heads```**BNNeck** (Luo et al., *Bag of Tricks*): the metric loss wants features spread ona hypersphere, the classification losses want them linearly separable and roughlyzero-centred — optimising both on the *same* vector makes them fight. Putting aBatchNorm between the two branches lets each get the geometry it wants. At testtime we rank with the post-BN feature, which is empirically the better retrievaldescriptor.**GeM pooling** interpolates between average (`p=1`) and max (`p→∞`) pooling with alearned `p`; on pedestrian crops it consistently beats plain average pooling.**`last_stride=1`** removes the final downsampling, so a 256×128 input yields a16×8 feature map instead of 8×4 — more spatial detail for free (it does costmemory, which is why it's a config flag).

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p)))
        self.eps = eps

    def forward(self, x):
        x = x.clamp(min=self.eps).pow(self.p)
        return F.adaptive_avg_pool2d(x, 1).pow(1.0 / self.p)


class AttributeHead(nn.Module):
    """One small head per attribute (kept from v1, with dropout as a module so it
    respects train()/eval())."""

    def __init__(self, in_dim, n_out, hidden=256, dropout=0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_out),
        )

    def forward(self, x):
        return self.block(x)


class ReIDNet(nn.Module):
    @staticmethod
    def _build_trunk(cfg):
        """torchvision ResNet, or its IBN-a variant when `cfg.ibn` and timm is
        available. IBN mixes InstanceNorm in the early layers (appearance
        invariance) with BatchNorm in the late ones (identity discrimination);
        it mostly helps cross-domain, so on single-domain Market-1501 expect a
        modest gain."""
        if cfg.ibn:
            try:
                import timm
                return timm.create_model(f'{cfg.backbone}_ibn_a', pretrained=True)
            except (ImportError, RuntimeError) as e:
                print(f'IBN requested but unavailable ({e}); falling back to torchvision')
        try:
            return getattr(models, cfg.backbone)(weights='DEFAULT')
        except TypeError:                                   # older torchvision
            return getattr(models, cfg.backbone)(pretrained=True)

    def __init__(self, attr_map, num_ids, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.attr_map = attr_map

        trunk = self._build_trunk(cfg)

        if cfg.last_stride == 1:
            # the stride-2 conv is conv1 in a BasicBlock (r18/r34) and conv2 in a
            # Bottleneck (r50), so just clear whichever one has it.
            block = trunk.layer4[0]
            for conv in (block.conv1, block.conv2):
                if conv.stride == (2, 2):
                    conv.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)

        self.trunk = nn.Sequential(*list(trunk.children())[:-2])
        trunk_dim = trunk.fc.in_features
        self.pool = GeM() if cfg.pooling == 'gem' else nn.AdaptiveAvgPool2d(1)

        if cfg.embedding_dim and cfg.embedding_dim != trunk_dim:
            self.embedding = nn.Sequential(nn.Linear(trunk_dim, cfg.embedding_dim, bias=False),
                                           nn.BatchNorm1d(cfg.embedding_dim),
                                           nn.LeakyReLU(inplace=True))
            feat_dim = cfg.embedding_dim
        else:
            self.embedding = nn.Identity()
            feat_dim = trunk_dim
        self.feat_dim = feat_dim

        # BNNeck: no bias, no learnable shift -> the classifier sees a whitened feature
        self.bottleneck = nn.BatchNorm1d(feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        nn.init.constant_(self.bottleneck.weight, 1.0)

        self.drop = nn.Dropout(cfg.dropout)
        self.id_classifier = nn.Linear(feat_dim, num_ids, bias=False)
        nn.init.normal_(self.id_classifier.weight, std=0.001)
        self.attr_heads = nn.ModuleList([AttributeHead(feat_dim, c) for c in attr_map])

    def forward(self, x, return_logits=True):
        f = self.pool(self.trunk(x)).flatten(1)
        f = self.embedding(f)
        f_bn = self.bottleneck(f) if self.cfg.bnneck else f

        if not return_logits:                # inference: retrieval descriptor only
            return f_bn

        h = self.drop(f_bn)
        return {
            'metric': f,                     # quadruplet loss operates here
            'retrieval': f_bn,               # what we rank with
            'id_logits': self.id_classifier(h),
            'attr_logits': [head(h) for head in self.attr_heads],
        }


model = ReIDNet(ATTR_MAP, NUM_TRAIN_IDS, cfg).to(DEVICE)
if cfg.channels_last:
    model = model.to(memory_format=torch.channels_last)

n_par = sum(p.numel() for p in model.parameters())
with torch.no_grad():
    dummy = torch.zeros(2, 3, *cfg.image_size, device=DEVICE)
    fmap = model.trunk(dummy)
print(f'{cfg.backbone}: {n_par / 1e6:.1f}M params | feature map {tuple(fmap.shape[1:])} '
      f'| descriptor {model.feat_dim}-d')

## 4. Losses### 4.1 Quadruplet loss with in-batch hard miningThe quadruplet loss of Chen et al. (2017) extends the triplet with a second termthat also pushes *apart* two different negative identities:$$L = \big[d_{ap} - d_{an} + \alpha_1\big]_+ \;+\; \big[d_{ap} - d_{n_1n_2} + \alpha_2\big]_+,\qquad \alpha_2 < \alpha_1$$The first term is the usual relative constraint; the second is an *absolute* one —it enlarges inter-class distances regardless of which anchor we started from,which is what makes the learned metric transfer to identities never seen duringtraining.Two changes with respect to v1:* **mining.** Within a PK batch we take, per anchor, the *hardest positive*  ($\max d_{ap}$) and the *hardest negative* ($\min d_{an}$), and for the second  term the closest pair of negatives from two different identities. v1 used random  ones, which are trivially satisfied after a few epochs → zero gradient.* **normalised features + soft margin.** Distances are computed on L2-normalised  embeddings (so they live in `[0, 2]` and the margins mean something, and they are  monotone in the cosine similarity actually used for ranking), and the hinge is  replaced by `softplus`, which never fully saturates.

In [ ]:
def pairwise_distance(x, squared=False, eps=1e-12):
    """Euclidean distance matrix, numerically safe for the diagonal."""
    dot = x @ x.t()
    sq = dot.diag()
    d2 = (sq.unsqueeze(0) - 2 * dot + sq.unsqueeze(1)).clamp(min=0)
    return d2 if squared else (d2 + eps).sqrt()


class QuadrupletLoss(nn.Module):
    def __init__(self, margin1=0.3, margin2=0.15, soft=True, normalize=True):
        super().__init__()
        self.m1, self.m2, self.soft, self.normalize = margin1, margin2, soft, normalize

    def _hinge(self, x, margin):
        return F.softplus(x + margin) if self.soft else F.relu(x + margin)

    def forward(self, features, labels):
        if self.normalize:
            features = F.normalize(features, dim=1)
        d = pairwise_distance(features)
        n = d.size(0)

        same = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask = same & ~torch.eye(n, dtype=torch.bool, device=d.device)
        neg_mask = ~same

        if not pos_mask.any() or not neg_mask.any():
            return d.sum() * 0.0, {}

        # batch-hard: furthest positive, closest negative
        d_ap = (d - 1e6 * (~pos_mask).float()).max(dim=1).values
        d_an = (d + 1e6 * (~neg_mask).float()).min(dim=1).values
        valid = pos_mask.any(1) & neg_mask.any(1)
        d_ap, d_an = d_ap[valid], d_an[valid]

        term1 = self._hinge(d_ap - d_an, self.m1)

        # second term: closest pair of negatives belonging to two different ids,
        # neither of which is the anchor's id.
        with torch.no_grad():
            hardest_neg = (d + 1e6 * (~neg_mask).float()).argmin(dim=1)
        nn_mask = neg_mask[hardest_neg] & neg_mask                 # differs from both ids
        d_nn = (d[hardest_neg] + 1e6 * (~nn_mask).float()).min(dim=1).values[valid]
        term2 = self._hinge(d_ap - d_nn, self.m2)

        loss = (term1 + term2).mean()
        stats = {
            'd_ap': d_ap.mean().item(),
            'd_an': d_an.mean().item(),
            'd_nn': d_nn.mean().item(),
            # fraction of anchors that still violate the margin: if this collapses
            # to 0 the loss has stopped teaching anything.
            'active': (d_ap - d_an + self.m1 > 0).float().mean().item(),
        }
        return loss, stats

### 4.2 Attribute loss with class weightsFor binary attributes we use `BCEWithLogitsLoss(pos_weight = n_neg / n_pos)`,capped at ×20 so that a very rare attribute does not swamp everything else. For`age` we use cross-entropy with inverse frequency weights and label smoothing.Weights are computed **on the training split only** — computing them on the fullframe would leak validation statistics.

In [ ]:
def attribute_weights(frame, cap=cfg.attr_weight_cap):
    pos_w, ce_w = {}, {}
    for i, a in enumerate(ATTRIBUTES):
        col = frame[a].to_numpy()
        if ATTR_MAP[i] == 1:
            pos = max(int((col == 1).sum()), 1)
            neg = max(int((col == 0).sum()), 1)
            pos_w[a] = float(min(neg / pos, cap))
        else:
            counts = np.bincount(col, minlength=ATTR_MAP[i]).astype(np.float64)
            counts[counts == 0] = 1
            w = counts.sum() / (len(counts) * counts)
            ce_w[a] = torch.tensor(np.minimum(w, cap), dtype=torch.float32)
    return pos_w, ce_w


class AttributesLoss(nn.Module):
    def __init__(self, frame, use_weights=True, label_smoothing=0.1, device=DEVICE):
        super().__init__()
        pos_w, ce_w = attribute_weights(frame) if use_weights else ({}, {})
        self.criteria = nn.ModuleList()
        for i, a in enumerate(ATTRIBUTES):
            if ATTR_MAP[i] == 1:
                w = torch.tensor([pos_w.get(a, 1.0)], device=device)
                self.criteria.append(nn.BCEWithLogitsLoss(pos_weight=w))
            else:
                w = ce_w.get(a)
                self.criteria.append(nn.CrossEntropyLoss(
                    weight=None if w is None else w.to(device),
                    label_smoothing=label_smoothing))

    def forward(self, logits_list, targets):
        total = 0.
        for i, logits in enumerate(logits_list):
            if logits.size(1) == 1:
                total = total + self.criteria[i](logits.float(),
                                                 targets[:, i].unsqueeze(1).float())
            else:
                total = total + self.criteria[i](logits.float(), targets[:, i])
        # FIX vs v1: mean over heads instead of sum, so the scale does not depend
        # on how many attributes there are and lambda_attr is interpretable.
        return total / len(logits_list)


attr_criterion = AttributesLoss(train_df, cfg.attr_class_weights, cfg.label_smoothing)
id_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
print('loss weights: attr', cfg.lambda_attr, '| metric', cfg.lambda_quadruplet,
      '| id', cfg.lambda_id)

### 4.3 Circle loss — the alternative to compare againstCircle loss (Sun et al., CVPR 2020, arXiv:2002.10857) replaces the hinge on`d_ap - d_an` with a *self-paced* weighting: each similarity score gets a weightproportional to how far it is from its own optimum, so the pairs that are alreadycorrect stop pulling and the ones that are badly wrong pull harder. The tripletfamily gives every violating pair the same gradient regardless of severity.This is here as an **isolated variant for the ablation table**(`Config(metric_loss='circle')`), not as a replacement for the quadruplet — theassignment is about the quadruplet network.> **Rescale the weight when you switch.** With `gamma=64` the circle loss comes> out around 30-40 while the quadruplet sits around 1-2, so at> `lambda_quadruplet=1.0` it drowns the attribute and ID terms. Use> `Config(metric_loss='circle', lambda_quadruplet=0.03)` to keep the three terms> comparable, otherwise the ablation row measures "circle loss only" rather than> "circle loss instead of quadruplet".

In [ ]:
class CircleLoss(nn.Module):
    """Circle loss on within-batch pairs (Sun et al., CVPR 2020)."""

    def __init__(self, m=0.25, gamma=64.0):
        super().__init__()
        self.m, self.gamma = m, gamma
        self.soft_plus = nn.Softplus()

    def forward(self, features, labels):
        features = F.normalize(features, dim=1)
        sim = features @ features.t()
        n = sim.size(0)

        same = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask = same & ~torch.eye(n, dtype=torch.bool, device=sim.device)
        neg_mask = ~same
        if not pos_mask.any() or not neg_mask.any():
            return sim.sum() * 0.0, {}

        sp = sim[pos_mask]        # similarities that should be high
        sn = sim[neg_mask]        # similarities that should be low

        # self-paced weights: distance of each score from its own optimum
        ap = torch.clamp_min(-sp.detach() + 1 + self.m, min=0.)
        an = torch.clamp_min(sn.detach() + self.m, min=0.)
        delta_p, delta_n = 1 - self.m, self.m

        logit_p = -ap * (sp - delta_p) * self.gamma
        logit_n = an * (sn - delta_n) * self.gamma

        loss = self.soft_plus(torch.logsumexp(logit_n, dim=0)
                              + torch.logsumexp(logit_p, dim=0))
        stats = {'s_pos': sp.mean().item(), 's_neg': sn.mean().item()}
        return loss, stats


def build_metric_criterion(cfg):
    if cfg.metric_loss == 'circle':
        return CircleLoss(cfg.circle_m, cfg.circle_gamma)
    return QuadrupletLoss(cfg.margin1, cfg.margin2, cfg.soft_margin)


quad_criterion = build_metric_criterion(cfg)
print('metric loss:', type(quad_criterion).__name__)

## 5. EvaluationTwo families of metrics:* **re-ID**: the assignment's `Evaluator.evaluate_map` (mAP over the top-`k`  ranked list) plus full mAP and CMC rank-1/5/10, which are the numbers the  literature reports.* **attributes**: per-attribute accuracy *and* balanced accuracy / F1, because on  attributes like `upyellow` plain accuracy is dominated by the majority class.Feature extraction optionally averages the descriptor of the image and of itsmirror (`flip_test`) — free ~1 point of mAP for one extra forward pass at testtime only.

In [ ]:
from typing import Dict, List, Set


class Evaluator:
    """Provided with the assignment - kept verbatim as the graded metric."""

    @staticmethod
    def evaluate_map(predictions: Dict[str, List], ground_truth: Dict[str, Set]):
        m_ap = 0.0
        for query, gt_set in ground_truth.items():
            if query not in predictions or not gt_set:
                continue
            current_ap, encountered = 0.0, 0
            delta_recall = 1.0 / len(gt_set)
            for idx, pred in enumerate(predictions[query]):
                if pred in gt_set:
                    encountered += 1
                    current_ap += (encountered / (idx + 1)) * delta_recall
            m_ap += current_ap
        return m_ap / max(len(ground_truth), 1)


@torch.no_grad()
def extract_features(model, loader, flip=None, device=DEVICE, desc='features'):
    flip = cfg.flip_test if flip is None else flip
    model.eval()
    out = []
    for batch in tqdm(loader, desc=desc, leave=False):
        x = (batch[0] if isinstance(batch, (list, tuple)) else batch)
        x = x.to(device, non_blocking=True)
        if cfg.channels_last:
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast('cuda', enabled=cfg.amp and torch.cuda.is_available()):
            f = model(x, return_logits=False).float()
            if flip:
                f = f + model(torch.flip(x, dims=[3]), return_logits=False).float()
        out.append(F.normalize(f, dim=1).cpu())
    return torch.cat(out)


def rank_gallery(query_features, gallery_features, k, chunk=512, re_rank=False):
    """Cosine similarity ranking. Normalised features -> a single matmul.

    With `re_rank=True` the ranking comes from the k-reciprocal re-ranked distance
    matrix instead (Zhong et al., CVPR 2017 — see re_ranking.py). It is a pure
    post-processing step, but it materialises (n_query+n_gallery)^2 matrices, so
    on the full 19,679-image test gallery it needs ~3 GB of RAM.
    """
    if re_rank:
        from re_ranking import re_ranking
        dist = torch.from_numpy(re_ranking(query_features, gallery_features,
                                           k1=cfg.re_rank_k1, k2=cfg.re_rank_k2,
                                           lambda_value=cfg.re_rank_lambda))
        v, i = (-dist).topk(min(k, dist.size(1)), dim=1)
        return i, -v

    idx, sims = [], []
    for s in range(0, query_features.size(0), chunk):
        sim = query_features[s:s + chunk] @ gallery_features.t()
        v, i = sim.topk(min(k, sim.size(1)), dim=1)
        idx.append(i); sims.append(v)
    return torch.cat(idx), torch.cat(sims)


def reid_metrics(query_features, gallery_features, query_ids, gallery_ids,
                 k=None, cmc_ranks=(1, 5, 10), re_rank=None):
    """Returns the assignment mAP@k plus full mAP and CMC."""
    k = k or cfg.map_rank
    re_rank = cfg.re_rank if re_rank is None else re_rank
    n_gallery = gallery_features.size(0)
    full_idx, _ = rank_gallery(query_features, gallery_features, n_gallery,
                               re_rank=re_rank)

    gallery_ids = np.asarray(gallery_ids)
    query_ids = np.asarray(query_ids)
    matches = gallery_ids[full_idx.numpy()] == query_ids[:, None]   # (Q, G) bool

    aps, cmc = [], np.zeros(max(cmc_ranks))
    for row in matches:
        rel = row.sum()
        if rel == 0:
            continue
        pos = np.flatnonzero(row)
        precision = (np.arange(len(pos)) + 1) / (pos + 1)
        aps.append(precision.mean())
        if pos[0] < len(cmc):           # first hit at rank pos[0]+1
            cmc[pos[0]:] += 1
    n_valid = len(aps)

    # the assignment metric: AP truncated to the top-k list
    top_k = full_idx[:, :k]
    gt = {q: set(np.flatnonzero(gallery_ids == i).tolist()) for q, i in enumerate(query_ids)}
    map_at_k = Evaluator.evaluate_map({q: r for q, r in enumerate(top_k.tolist())}, gt)

    res = {f'mAP@{k}': map_at_k, 'mAP': float(np.mean(aps)) if aps else 0.0}
    for r in cmc_ranks:
        res[f'rank-{r}'] = float(cmc[r - 1] / max(n_valid, 1))
    return res, top_k

In [ ]:
@torch.no_grad()
def attribute_metrics(model, loader, device=DEVICE):
    """Per-attribute accuracy, balanced accuracy and macro-F1."""
    model.eval()
    preds = [[] for _ in ATTRIBUTES]
    gts = []
    for imgs, attrs, _ in tqdm(loader, desc='attrs', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        if cfg.channels_last:
            imgs = imgs.contiguous(memory_format=torch.channels_last)
        with torch.autocast('cuda', enabled=cfg.amp and torch.cuda.is_available()):
            out = model(imgs)
        for i, logits in enumerate(out['attr_logits']):
            p = ((logits.float().squeeze(1) > 0).long() if logits.size(1) == 1
                 else logits.float().argmax(1))
            preds[i].append(p.cpu())
        gts.append(attrs)

    gts = torch.cat(gts).numpy()
    rows = []
    for i, a in enumerate(ATTRIBUTES):
        p = torch.cat(preds[i]).numpy()
        y = gts[:, i]
        acc = (p == y).mean()
        classes = np.unique(y)
        # balanced accuracy = mean recall over the classes actually present
        recalls = [(p[y == c] == c).mean() for c in classes]
        f1s = []
        for c in classes:
            tp = ((p == c) & (y == c)).sum()
            fp = ((p == c) & (y != c)).sum()
            fn = ((p != c) & (y == c)).sum()
            f1s.append(2 * tp / max(2 * tp + fp + fn, 1))
        # mA (mean Accuracy), the standard pedestrian-attribute-recognition
        # metric: the mean of the per-class recalls. For a binary attribute this
        # is exactly 0.5 * (TP/P + TN/N), so a head that always answers with the
        # majority class scores 50, not 97.
        rows.append(dict(attribute=a, accuracy=acc * 100,
                         mA=float(np.mean(recalls)) * 100,
                         macro_f1=float(np.mean(f1s)) * 100))
    table = pd.DataFrame(rows)
    summary = dict(attr_acc=table.accuracy.mean(),
                   attr_mA=table.mA.mean(),
                   attr_f1=table.macro_f1.mean())
    return summary, table


def evaluate(model, verbose=False, re_rank=None):
    qf = extract_features(model, query_loader, desc='query')
    gf = extract_features(model, gallery_loader, desc='gallery')
    reid, _ = reid_metrics(qf, gf, query_df.ID.to_numpy(), gallery_df.ID.to_numpy(),
                           re_rank=re_rank)
    attr, table = attribute_metrics(model, val_loader)
    if verbose:
        display(table.sort_values('macro_f1').style.format('{:.1f}', subset=table.columns[1:]))
    return {**reid, **attr}, table

## 6. Training loop* **AdamW** with a 10× smaller learning rate on the pretrained trunk than on the  freshly initialised heads — otherwise the random heads' large early gradients  destroy the ImageNet features in the first few hundred steps.* **linear warmup + cosine decay**, per step. Warmup matters a lot with BNNeck and  metric losses (see *Bag of Tricks*); v1 used a constant LR for 50 epochs.* **AMP** with `GradScaler`, `channels_last`, optional gradient accumulation.* Every epoch logs the quadruplet's `active` fraction: when it approaches 0 the  mining has run out of hard examples in the batch and it's time to raise `P` or  the margin.

In [ ]:
def build_optimizer(model, cfg):
    trunk_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (trunk_params if name.startswith('trunk') else head_params).append(p)
    return torch.optim.AdamW([
        {'params': trunk_params, 'lr': cfg.lr * cfg.lr_backbone_mult},
        {'params': head_params, 'lr': cfg.lr},
    ], weight_decay=cfg.weight_decay)


class WarmupCosine:
    def __init__(self, optimizer, warmup_steps, total_steps, min_scale=0.01):
        self.opt = optimizer
        self.base = [g['lr'] for g in optimizer.param_groups]
        self.warmup, self.total, self.min_scale = max(warmup_steps, 1), total_steps, min_scale
        self.step_n = 0

    def step(self):
        self.step_n += 1
        if self.step_n <= self.warmup:
            scale = self.step_n / self.warmup
        else:
            t = (self.step_n - self.warmup) / max(self.total - self.warmup, 1)
            scale = self.min_scale + (1 - self.min_scale) * 0.5 * (1 + math.cos(math.pi * min(t, 1)))
        for g, b in zip(self.opt.param_groups, self.base):
            g['lr'] = b * scale
        return scale

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    if hasattr(loader.sampler, 'set_epoch'):
        loader.sampler.set_epoch(epoch)

    sums = defaultdict(float)
    n = 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'epoch {epoch + 1}', leave=False)

    for step, (imgs, attrs, labels) in enumerate(pbar):
        imgs = imgs.to(DEVICE, non_blocking=True)
        if cfg.channels_last:
            imgs = imgs.contiguous(memory_format=torch.channels_last)
        attrs = attrs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast('cuda', enabled=cfg.amp and torch.cuda.is_available()):
            out = model(imgs)
            l_attr = attr_criterion(out['attr_logits'], attrs)
            l_quad, qstats = quad_criterion(out['metric'], labels)
            l_id = id_criterion(out['id_logits'].float(), labels)
            loss = (cfg.lambda_attr * l_attr
                    + cfg.lambda_quadruplet * l_quad
                    + cfg.lambda_id * l_id)

        scaler.scale(loss / cfg.grad_accum).backward()

        if (step + 1) % cfg.grad_accum == 0:
            if cfg.grad_clip:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        n += 1
        sums['loss'] += loss.item()
        sums['attr'] += float(l_attr)
        sums['quad'] += float(l_quad)
        sums['id'] += float(l_id)
        for k, v in qstats.items():
            sums[k] += v
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{sums["loss"] / n:.3f}',
                             quad=f'{sums["quad"] / n:.3f}',
                             active=f'{sums.get("active", 0) / n:.2f}',
                             lr=f'{optimizer.param_groups[-1]["lr"]:.2e}')

    return {k: v / max(n, 1) for k, v in sums.items()}

In [ ]:
def fit(model, cfg, resume=False):
    optimizer = build_optimizer(model, cfg)
    steps_per_epoch = max(len(train_loader) // cfg.grad_accum, 1)
    scheduler = WarmupCosine(optimizer, cfg.warmup_epochs * steps_per_epoch,
                             cfg.epochs * steps_per_epoch)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.amp and torch.cuda.is_available())

    start_epoch, best, history, stale = 0, -1.0, [], 0
    if resume and os.path.exists(cfg.checkpoint):
        ck = torch.load(cfg.checkpoint, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        scheduler.step_n = ck['sched_step']
        start_epoch, best, history = ck['epoch'] + 1, ck['best'], ck['history']
        print(f'resumed from epoch {start_epoch} (best mAP@{cfg.map_rank} {best:.4f})')

    for epoch in range(start_epoch, cfg.epochs):
        tr = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)

        row = {'epoch': epoch + 1, **{f'train_{k}': v for k, v in tr.items()}}
        if (epoch + 1) % cfg.eval_every == 0 or epoch == cfg.epochs - 1:
            metrics, _ = evaluate(model)
            row.update(metrics)
            key = metrics[f'mAP@{cfg.map_rank}']
            improved = key > best
            if improved:
                best, stale = key, 0
                torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                            'scaler': scaler.state_dict(), 'sched_step': scheduler.step_n,
                            'epoch': epoch, 'best': best, 'history': history + [row],
                            'cfg': asdict(cfg), 'attr_map': ATTR_MAP,
                            'attributes': ATTRIBUTES, 'num_ids': NUM_TRAIN_IDS},
                           cfg.checkpoint)
            else:
                stale += 1
            print(f"epoch {epoch + 1:3d} | loss {tr['loss']:.3f} "
                  f"(attr {tr['attr']:.3f} metric {tr['quad']:.3f} id {tr['id']:.3f}) "
                  f"| active {tr.get('active', 0):.2f} "
                  f"| mAP@{cfg.map_rank} {key:.4f} "
                  f"mAP {metrics['mAP']:.4f} r1 {metrics['rank-1']:.4f} "
                  f"| attr mA {metrics['attr_mA']:.1f} F1 {metrics['attr_f1']:.1f}"
                  f"{'  *best*' if improved else ''}")

            # early stopping: the cosine schedule means late epochs change little,
            # and the attribute heads overfit long before the metric branch does
            if cfg.patience and epoch + 1 >= cfg.min_epochs and stale >= cfg.patience:
                print(f'early stop: {stale} evaluations without improvement')
                history.append(row)
                break
        else:
            print(f"epoch {epoch + 1:3d} | loss {tr['loss']:.3f}")
        history.append(row)

    pd.DataFrame(history).to_csv(os.path.join(cfg.out_dir, 'history.csv'), index=False)
    return pd.DataFrame(history), best

### Sanity check before the real runBefore spending an hour on the GPU: overfit a handful of identities. If the lossdoes not go to ~0 and mAP to ~1 on 40 identities, something is wired wrong and noamount of epochs will fix it. This is the check that would have caught v1'simage/label misalignment in two minutes.

In [ ]:
# Set to True for the smoke test (a couple of minutes).
RUN_SANITY_CHECK = False

if RUN_SANITY_CHECK:
    sanity_cfg = Config(**{**asdict(cfg), 'epochs': 3, 'warmup_epochs': 1,
                           'backbone': 'resnet18', 'p_identities': 8,
                           'out_dir': 'runs/sanity', 'checkpoint': 'runs/sanity/best.pt'})
    os.makedirs(sanity_cfg.out_dir, exist_ok=True)
    sub_ids = sorted(train_df.label.unique())[:8]
    sub = train_df[train_df.label.isin(sub_ids)].reset_index(drop=True)
    sanity_ds = MarketDataset(os.path.join(cfg.data_root, 'train'), sub, train_tfms)
    sanity_sampler = PKSampler(sub.label.to_numpy(), 8, 4, cfg.seed)
    sanity_loader = DataLoader(sanity_ds, batch_size=32, sampler=sanity_sampler,
                               drop_last=True, num_workers=2)
    smodel = ReIDNet(ATTR_MAP, NUM_TRAIN_IDS, sanity_cfg).to(DEVICE)
    sopt = build_optimizer(smodel, sanity_cfg)
    ssched = WarmupCosine(sopt, 5, 3 * len(sanity_loader))
    sscaler = torch.amp.GradScaler('cuda', enabled=sanity_cfg.amp and torch.cuda.is_available())
    for e in range(6):
        st = train_one_epoch(smodel, sanity_loader, sopt, ssched, sscaler, e)
        print(f"  sanity epoch {e + 1}: loss {st['loss']:.3f} quad {st['quad']:.3f} "
              f"id {st['id']:.3f} active {st.get('active', 0):.2f}")
    print('loss and quad should both be clearly decreasing.')

In [ ]:
history, best_map = fit(model, cfg, resume=False)
print(f'best mAP@{cfg.map_rank} = {best_map:.4f}  ->  {cfg.checkpoint}')
history.tail()

In [ ]:
h = pd.read_csv(os.path.join(cfg.out_dir, 'history.csv'))
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))
h.plot(x='epoch', y=['train_loss', 'train_attr', 'train_quad', 'train_id'], ax=axs[0],
       title='losses')
h.dropna(subset=[f'mAP@{cfg.map_rank}']).plot(
    x='epoch', y=[f'mAP@{cfg.map_rank}', 'mAP', 'rank-1'], ax=axs[1], title='re-ID')
h.dropna(subset=['attr_acc']).plot(x='epoch', y=['attr_acc', 'attr_mA', 'attr_f1'],
                                   ax=axs[2], title='attributes')
for ax in axs:
    ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. Final evaluation on the held-out identities

In [ ]:
ck = torch.load(cfg.checkpoint, map_location=DEVICE, weights_only=False)
model = ReIDNet(ck['attr_map'], ck['num_ids'], cfg).to(DEVICE)
model.load_state_dict(ck['model'])
if cfg.channels_last:
    model = model.to(memory_format=torch.channels_last)
model.eval()
print(f"loaded epoch {ck['epoch'] + 1} | best mAP@{cfg.map_rank} {ck['best']:.4f}")

metrics, attr_table = evaluate(model, verbose=True)
for k, v in metrics.items():
    print(f'  {k:12s} {v:.4f}')

The per-attribute table above is the interesting one: sort by `macro_f1` and therare attributes (`hat`, `uppurple`, `downpink`, ...) sit at the bottom. Their plain*accuracy* is in the high nineties because they are almost always negative — andthat is exactly the number v1 reported and averaged into a single "totalaccuracy".

In [ ]:
worst = attr_table.sort_values('macro_f1').head(8)
best_a = attr_table.sort_values('macro_f1').tail(8)
fig, ax = plt.subplots(figsize=(7, 5))
pd.concat([worst, best_a]).set_index('attribute')[['accuracy', 'macro_f1']].plot.barh(ax=ax)
ax.set_title('accuracy vs macro-F1: where the metric lies to you')
plt.tight_layout(); plt.show()

### k-reciprocal re-rankingTwo images that are *each other's* k-nearest neighbours are far more likely to bethe same person than two that are neighbours in one direction only. k-reciprocalencoding (Zhong et al., CVPR 2017) turns that observation into a re-rankeddistance: each image becomes a sparse vector over its expanded k-reciprocal set,and images are re-ordered by the Jaccard distance between those vectors, blendedwith the original one.It is pure post-processing — no training, no extra forward passes — and it is thestrongest single post-hoc gain in the re-ID literature (+8-10 mAP on the standardMarket-1501 protocol; every row of the leaderboard has an "(RK)" variant).**Two caveats for the report.** (1) Our gallery is a *subset of the trainingidentities*, much smaller than the 19,732-image Market-1501 gallery, and the gainof re-ranking shrinks with the gallery size — do not expect the full +8. (2) Theassignment's metric truncates the ranked list at 20 while a query has ~20 relevantgallery images, so `mAP@20` is bounded well below 1 by construction and compressesdifferences. Report `mAP` (full) alongside it.

In [ ]:
qf_rr = extract_features(model, query_loader, desc='query')
gf_rr = extract_features(model, gallery_loader, desc='gallery')

plain, _ = reid_metrics(qf_rr, gf_rr, query_df.ID.to_numpy(), gallery_df.ID.to_numpy(),
                        re_rank=False)
# k1 is scaled down: 20 is tuned for a 19.7k gallery, ours is ~10x smaller
reranked, _ = reid_metrics(qf_rr, gf_rr, query_df.ID.to_numpy(), gallery_df.ID.to_numpy(),
                           re_rank=True)

comparison = pd.DataFrame([plain, reranked], index=['cosine', 'k-reciprocal RK'])
display(comparison.style.format('{:.4f}'))

### Ablation tableEach row adds one component to the previous one, so the deltas are attributable.`run_ablation` is deliberately explicit rather than clever: every row is a full`Config`, so you can run a subset, interrupt, and resume.Expected from the literature: the two big jumps are **BNNeck + ID loss** and**PK sampler + batch-hard mining**. If you do not observe them, there is still abug — that is the point of the table.Running all rows takes several hours; `ABLATION_ROWS` selects which ones to run.

In [ ]:
ABLATION = {
    'baseline (fixed v1 recipe)': dict(
        backbone='resnet18', last_stride=2, pooling='avg', bnneck=False,
        lambda_id=0.0, attr_class_weights=False, p_identities=12, k_instances=4),
    '+ 256x128 + warmup/cosine': dict(
        backbone='resnet18', last_stride=2, pooling='avg', bnneck=False,
        lambda_id=0.0, attr_class_weights=False),
    '+ BNNeck + ID loss + smoothing': dict(
        backbone='resnet18', last_stride=2, pooling='avg'),
    '+ GeM + last-stride=1': dict(backbone='resnet18'),
    '+ pos_weight BCE': dict(backbone='resnet18', attr_class_weights=True),
    '+ resnet50': dict(backbone='resnet50'),
    '(variant) circle loss': dict(backbone='resnet50', metric_loss='circle',
                                  lambda_quadruplet=0.03),
    '(variant) IBN-a': dict(backbone='resnet50', ibn=True),
}

# which rows to actually run in this session
ABLATION_ROWS = []          # e.g. ['+ BNNeck + ID loss + smoothing']


def run_ablation(rows, base=cfg, epochs=None):
    results = {}
    for name in rows:
        overrides = dict(ABLATION[name])
        slug = ''.join(c if c.isalnum() else '_' for c in name)[:40]
        run_cfg = Config(**{**asdict(base), **overrides,
                            'epochs': epochs or base.epochs,
                            'out_dir': f'runs/ablation/{slug}',
                            'checkpoint': f'runs/ablation/{slug}/best.pt'})
        os.makedirs(run_cfg.out_dir, exist_ok=True)
        print(f'\n=== {name} ===')

        seed_everything(run_cfg.seed)
        net = ReIDNet(ATTR_MAP, NUM_TRAIN_IDS, run_cfg).to(DEVICE)
        if run_cfg.channels_last:
            net = net.to(memory_format=torch.channels_last)

        global cfg, quad_criterion, attr_criterion
        prev_cfg, prev_metric, prev_attr = cfg, quad_criterion, attr_criterion
        cfg = run_cfg
        quad_criterion = build_metric_criterion(run_cfg)
        attr_criterion = AttributesLoss(train_df, run_cfg.attr_class_weights,
                                        run_cfg.label_smoothing)
        try:
            _, best = fit(net, run_cfg)
            metrics, _ = evaluate(net)
            results[name] = {f'mAP@{run_cfg.map_rank}': best, **metrics}
        finally:
            cfg, quad_criterion, attr_criterion = prev_cfg, prev_metric, prev_attr

    return pd.DataFrame(results).T


if ABLATION_ROWS:
    ablation_table = run_ablation(ABLATION_ROWS)
    display(ablation_table)
    ablation_table.to_csv('runs/ablation_table.csv')
else:
    print('set ABLATION_ROWS to run rows; available:')
    for k in ABLATION:
        print(' -', k)

## 8. Task 1 — attribute predictions on `test/`

In [ ]:
@torch.no_grad()
def predict_attributes(model, loader, device=DEVICE):
    model.eval()
    chunks = []
    for imgs in tqdm(loader, desc='predict', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        if cfg.channels_last:
            imgs = imgs.contiguous(memory_format=torch.channels_last)
        with torch.autocast('cuda', enabled=cfg.amp and torch.cuda.is_available()):
            out = model(imgs)
        cols = []
        for logits in out['attr_logits']:
            p = ((logits.float().squeeze(1) > 0).long() if logits.size(1) == 1
                 else logits.float().argmax(1))
            cols.append(p.unsqueeze(1))
        chunks.append(torch.cat(cols, 1).cpu())
    return torch.cat(chunks).numpy()


preds = predict_attributes(model, test_loader)
# the index comes from the dataset's own sorted file list, so rows and filenames
# cannot drift apart (v1 indexed with an independent, unsorted os.listdir call)
submission = pd.DataFrame(preds, index=test_ds.names, columns=ATTRIBUTES)
submission.index.name = 'image_name'
submission.to_csv('classification_test.csv')
print(submission.shape)
submission.head()

## 9. Task 2 — re-identification ranking

> Set `SUBMIT_WITH_RERANK = True` to produce the submission with k-reciprocal> re-ranking. On the full test set that means a 21,927 x 21,927 distance matrix,> i.e. roughly 3 GB of **system** RAM (not VRAM) and a few minutes of CPU. Check> on the validation split first whether it actually helps at this gallery size.

In [ ]:
SUBMIT_WITH_RERANK = False

test_features = extract_features(model, test_loader, desc='test gallery')
query_features = extract_features(model, queries_loader, desc='queries')
top_k, top_sims = rank_gallery(query_features, test_features, cfg.map_rank,
                               re_rank=SUBMIT_WITH_RERANK)

with open('reid_test.txt', 'w') as f:                # 'w', not 'a'
    for i, ids in enumerate(top_k.tolist()):
        f.write(f"{queries_ds.names[i]}: {', '.join(test_ds.names[j] for j in ids)}\n")

print(f'{top_k.shape[0]} queries x top-{top_k.shape[1]}')
print(open('reid_test.txt').readline())

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download('classification_test.csv')
    files.download('reid_test.txt')
else:
    print('written: classification_test.csv, reid_test.txt')

## 10. Qualitative resultsRetrieval on the held-out **validation** identities, where we know the groundtruth: green = same identity, red = wrong. This is the only honest qualitativecheck — the `test/` folder has no labels, so a grid of plausible-lookingneighbours there proves nothing.

In [ ]:
qf = extract_features(model, query_loader, desc='query')
gf = extract_features(model, gallery_loader, desc='gallery')
vtop, _ = rank_gallery(qf, gf, 10)

n_show, k_show = 4, 10
fig, axs = plt.subplots(n_show, k_show + 1, figsize=(1.1 * (k_show + 1), 2.4 * n_show))
rng = random.Random(cfg.seed)
for r, qi in enumerate(rng.sample(range(len(query_df)), n_show)):
    q_id = query_df.ID[qi]
    axs[r, 0].imshow(Image.open(os.path.join(cfg.data_root, 'train',
                                             query_df.image_name[qi])))
    axs[r, 0].set_title(f'query {q_id}', fontsize=7)
    axs[r, 0].axis('off')
    for c, gi in enumerate(vtop[qi].tolist()):
        ok = gallery_df.ID[gi] == q_id
        axs[r, c + 1].imshow(Image.open(os.path.join(cfg.data_root, 'train',
                                                     gallery_df.image_name[gi])))
        axs[r, c + 1].axis('off')
        for s in ('bottom', 'top', 'left', 'right'):
            axs[r, c + 1].spines[s].set_visible(True)
        axs[r, c + 1].patch.set_edgecolor('green' if ok else 'red')
        axs[r, c + 1].set_title('✓' if ok else '✗', fontsize=8,
                                color='green' if ok else 'red')
plt.suptitle('validation retrieval — rank 1..10', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# attribute predictions on a few random test images
idx = random.Random(cfg.seed).sample(range(len(test_ds.names)), 3)
for i in idx:
    im = Image.open(os.path.join(cfg.data_root, 'test', test_ds.names[i]))
    row = submission.iloc[i]
    active = [a for a in ATTRIBUTES if a != 'age' and row[a] == 1]
    plt.figure(figsize=(1.6, 3.2))
    plt.imshow(im); plt.axis('off')
    plt.title(f"age={row['age']}\n" + '\n'.join(active), fontsize=7)
    plt.show()

## 11. Hardware budget — running this on an 8 GB laptop GPUMeasured activation memory dominates; parameters and optimiser state are small(ResNet50 ≈ 25 M params → ~100 MB fp32 weights + ~200 MB AdamW state).| preset | images / step | resolution | AMP | peak VRAM ||---|---|---|---|---|| `resnet18`, `last_stride=2`, P·K = 24·4 | 96 | 256×128 | on | ~1.9 GB || `resnet50`, `last_stride=1`, P·K = 12·4 (**default**) | 48 | 256×128 | on | ~4.0 GB || `resnet50`, `last_stride=1`, P·K = 16·4 | 64 | 256×128 | on | ~5.2 GB || `resnet50`, `last_stride=1`, P·K = 12·4 | 48 | 256×128 | off | ~7.4 GB → very tight || v1 recipe (`resnet18`, 4 fwd of 48 imgs, 224×224, fp32) | 192 | 224×224 | off | ~6.5 GB, very tight |Run the cell below to measure it on *your* GPU before launching the full training.If you do hit an OOM: lower `p_identities` first (keep `k_instances=4`, it is whatmakes mining work), then set `last_stride=2`, then drop to `resnet34`. Use`grad_accum` to keep the effective batch size while halving the memory.

In [ ]:
def measure_peak_memory(cfg_, steps=3):
    if not torch.cuda.is_available():
        print('no CUDA here — run this on the laptop'); return
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m = ReIDNet(ATTR_MAP, NUM_TRAIN_IDS, cfg_).to(DEVICE)
    if cfg_.channels_last:
        m = m.to(memory_format=torch.channels_last)
    opt = build_optimizer(m, cfg_)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg_.amp)
    x = torch.randn(cfg_.batch_size, 3, *cfg_.image_size, device=DEVICE)
    if cfg_.channels_last:
        x = x.contiguous(memory_format=torch.channels_last)
    a = torch.randint(0, 2, (cfg_.batch_size, len(ATTRIBUTES)), device=DEVICE)
    a[:, 0] = torch.randint(0, 4, (cfg_.batch_size,), device=DEVICE)
    lbl = torch.arange(cfg_.batch_size, device=DEVICE) // cfg_.k_instances

    m.train()
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=cfg_.amp):
            out = m(x)
            loss = (attr_criterion(out['attr_logits'], a)
                    + quad_criterion(out['metric'], lbl)[0]
                    + id_criterion(out['id_logits'].float(), lbl))
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    peak = torch.cuda.max_memory_allocated() / 2**30
    print(f'{cfg_.backbone} stride{cfg_.last_stride} '
          f'{cfg_.p_identities}x{cfg_.k_instances}={cfg_.batch_size} imgs '
          f'{cfg_.image_size} amp={cfg_.amp} -> peak {peak:.2f} GiB')
    del m, opt, x
    torch.cuda.empty_cache()
    return peak


measure_peak_memory(cfg)

### Throughput noteOn an RTX 4070 Laptop (8 GB, ~35 % of a desktop 4070's power budget) the defaultpreset runs at roughly **2–3 min/epoch**, so 30 epochs is about **1–1.5 h**. Twothings usually become the bottleneck before the GPU does:1. **JPEG decoding.** 12,989 tiny 64×128 jpegs per epoch: keep   `num_workers` at 6–8 and `pin_memory=True`. If the GPU sits below ~80 %   utilisation (`nvidia-smi dmon`), you are CPU-bound, not GPU-bound.2. **Thermal/power throttling.** Laptop GPUs sustain far less than their boost   clock. Plug in the charger and pick the performance power profile, otherwise   expect 30–40 % fewer images/s after the first few minutes.For comparison, v1's recipe did 4 forward passes of 48 images at 224×224 in fp32per step — about **6× the FLOPs per identity seen** — and re-extracted the fullgallery+query features (and the whole training set!) inside `test()` on everyepoch.

## 12. What the numbers do and do not meanThree things must be stated in the report before any comparison with theliterature, otherwise the results read as far better or far worse than they are.**1. This `mAP@20` is not the Market-1501 `mAP`.** Three differences, all of whichpush it *down* relative to the published protocol:* the gallery is a subset of the *training* images (~2.4k), not the official  19,732-image gallery;* the standard protocol removes "junk" images — same identity *and* same camera as  the query — because matching them rewards background similarity rather than  identity. The assignment's `Evaluator` does not, and neither do we, so as not to  change the graded metric;* the ranked list is truncated at 20 while a query has ~20 relevant gallery  images, so the achievable AP is bounded well below 1 by construction.Quoting "89.9% mAP on Market-1501" next to this number compares two differentquantities. Report full `mAP` and `rank-1` alongside it.**2. The attribute labels are annotated per identity, not per image.** Every imageof a person carries the same 29 labels, so the number of independent attributesamples is 751, not 12,989 — and the model can reach a good score by recognisingthe *person* and recalling their attributes, rather than by reading the attributeoff the pixels. This makes the task easier than standard pedestrian attributerecognition (PA-100K, PETA) and is why the identity split matters so much.**3. Attribute accuracy is not attribute performance.** `hat` is positive on 2.7%of the images: always answering "no" scores 97.3% accuracy and 50.0 mA. Lead with**mA** and macro-F1.